# On Cue: run notebook

Can a model predict, from strictly causal class talk and context up to second *t*, that Brady
raises his hand in the next 15 seconds? Design: `design/on_cue_spec.md` (plus
`design/amendments.md`); write-up: `README.md`; pre-registration: `PLAN.md` (tag `prereg`).

**How this notebook runs.** Each pipeline stage is a CLI step (`python -m oncue <stage>`).
The four `RERUN_*` flags below default to `False`. With a flag off, its cell only prints the
commands it would run. Every later cell then loads either the local tables in `data/` (Brady's
machine) or the committed, privacy-safe bundle in `results/` (HMAC class ids, no words). So the
metric, figure and table cells re-run for a grader who has no data.

**Privacy.** Nothing here prints classmates' words or names. Figures are aggregate-only
(`plots.assert_aggregate`), and `python -m oncue privacy` must report 0 hits before anything is
committed.

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import SVG, Markdown, display

PIPELINE = Path.cwd() if (Path.cwd() / "oncue").is_dir() else Path.cwd() / "pipeline1"
if str(PIPELINE) not in sys.path:
    sys.path.insert(0, str(PIPELINE))

from oncue import config, plots, qa, report  # noqa: E402

RERUN_HARVEST = False     # Forum sync, class_edit_page, PCW (needs pipeline1/.env; network)
RERUN_TRANSCRIBE = False  # Scribe ASR per class, then words / events / pcw / frames / embed
RERUN_SCAN = False        # hand scan (signed recording URLs), track, episodes, retime, qa, labels
RERUN_MODELS = False      # features, train, evaluate, controls, figures, results bundle
RUN_TESTS = True          # the causal-contract tests (offline, synthetic data only)
SAVE_FIGURES = False      # overwrite figures/ (committed, privacy-scanned) from this notebook

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)


def oncue(*args, rerun: bool):
    # Print one CLI step; run it only when its RERUN flag is on.
    print("$ python -m oncue " + " ".join(args))
    if rerun:
        subprocess.run([sys.executable, "-m", "oncue", *args], cwd=PIPELINE, check=True)


def local(name: str):
    # A local table from data/tables (None on a machine without data/).
    path = config.table_path(name)
    return pd.read_parquet(path) if path.exists() else None


def local_json(path: Path):
    return json.loads(path.read_text()) if path.exists() else None


def show(fig_or_none, name: str):
    # Show a freshly drawn figure (saved to figures/ when SAVE_FIGURES), or the committed
    # figures/{name}.svg when this machine lacks the inputs.
    if fig_or_none is not None:
        if SAVE_FIGURES:
            plots.save(fig_or_none, name)
        display(fig_or_none)
        return
    svg = config.FIGURES_DIR / f"{name}.svg"
    if svg.exists():
        display(SVG(filename=str(svg)))
    else:
        print(f"({name}: no inputs here and no committed figure)")


LOCAL = config.TABLES_DIR.exists()
bundle = report.load_bundle()
results = bundle.results if bundle.results is not None else local("results")
print(f"local data: {LOCAL}; committed bundle: {not bundle.empty}")

## 0. Provenance

In [ ]:
prov = report.provenance()
display(pd.Series({k: v for k, v in prov.items() if k != "packages"}, name="value").to_frame())
display(pd.Series(prov["packages"], name="version").to_frame())
counts = {"results rows": None if results is None else len(results),
          "oof conditions": len(bundle.oofs),
          "bids": None if bundle.bids is None else len(bundle.bids)}
if LOCAL:
    for name in ("classes", "words", "points", "features", "hand_episodes"):
        t = local(name)
        counts[name] = None if t is None else len(t)
display(pd.Series(counts, name="rows").to_frame())
display(Markdown("Scribe must be a pushed SHA for final results (A7); an editable install is "
                 "flagged above. Ids in `results/` are HMAC-SHA256 with a secret salt."))

### Pipeline steps (printed unless a RERUN flag is on)

The order of a real run. `python -m oncue e2e` runs every step on the synthetic world in a
sandbox (no network, no real data) and is what `tests/test_end_to_end.py` exercises.

In [ ]:
oncue("harvest", "--sync", rerun=RERUN_HARVEST)
oncue("select", rerun=RERUN_HARVEST)
oncue("harvest", "--phase", "A", rerun=RERUN_HARVEST)            # class_edit_page + PCW, once

oncue("transcribe", "--phase", "A", rerun=RERUN_TRANSCRIBE)      # never passes --hand-log
for stage in ("words", "events", "pcw", "frames"):
    oncue(stage, rerun=RERUN_TRANSCRIBE)
oncue("embed", "--edu", rerun=RERUN_TRANSCRIBE)

for stage in ("scan", "track", "episodes", "retime", "qa", "labels"):
    oncue(stage, *(("--workers", "4") if stage == "scan" else ()), rerun=RERUN_SCAN)

oncue("features", rerun=RERUN_MODELS)
oncue("train", "--tier", "must", "--up-to", rerun=RERUN_MODELS)
oncue("evaluate", "--tier", "must", "--up-to", rerun=RERUN_MODELS)
oncue("controls", rerun=RERUN_MODELS)
oncue("figures", rerun=RERUN_MODELS)
oncue("results-bundle", rerun=RERUN_MODELS)
oncue("privacy", rerun=RERUN_MODELS)

## 1. Data funnel

In [ ]:
funnel = bundle.funnel
classes = local("classes")
if classes is not None:
    funnel = qa.funnel_table(classes, local("points"))
if funnel is not None:
    display(funnel)
    show(plots.fig_funnel(funnel), "funnel")
else:
    print("no funnel yet (run `oncue qa` and `oncue results-bundle`)")
if classes is not None:
    sel = classes[classes["phase"].notna()]
    display(pd.crosstab(sel["course_code"], sel["pcw_status"].fillna("not harvested"),
                        margins=True).rename_axis(index="course", columns="PCW status"))

## 2. Label validation (G2)

The confirm-set audit (contact sheets and recall clips, §2.7) is graded by hand in
`audit/*_manifest.csv` (the `verdict` column), which is local and deleted on Mon 19 Oct. Only
the counts below are reported.

In [ ]:
from oncue import hands

audit = sorted(config.AUDIT_DIR.glob("*manifest*.csv")) if config.AUDIT_DIR.exists() else []
for path in audit:
    m = pd.read_csv(path)
    v = m["verdict"].dropna().astype(str).str.lower() if "verdict" in m else pd.Series(dtype=str)
    k, n = int(v.isin(["ok", "yes", "correct", "1", "true"]).sum()), int(len(v))
    lo, hi = config.wilson_interval(k, n) if n else (np.nan, np.nan)
    print(f"{path.name}: {k}/{n} correct, Wilson 95% [{lo:.2f}, {hi:.2f}]")
bids = bundle.bids if bundle.bids is not None else local("hand_episodes")
if bids is not None:
    display(bids.groupby("ep_type").size().rename("episodes").to_frame())
    b = bids[bids["ep_type"] == config.EP_BID]
    display(b["outcome"].value_counts(dropna=False).rename("bids").to_frame())
    display(b["call_latency_s"].describe().to_frame("call latency (s)"))
eps, hf = local("hand_episodes"), local("hand_frames")
if eps is not None:
    print(f"left-censored share: {hands.left_censored_share(eps):.3f}")
    print(hands.vote_diagnostics(eps, hf))
onsets = local("speech_onsets")
if onsets is not None and classes is not None:
    share = onsets.merge(classes[["class_id", "section_id"]], on="class_id") \
        .groupby("section_id")["hand_preceded"].mean().rename("hand_preceded_share")
    display(share.describe().to_frame())

## 3. Clock: calibration, A/V lag, P-clock

In [ ]:
if classes is not None:
    sel = classes[classes["phase"].notna()]
    display(sel["cal_method"].value_counts(dropna=False).rename("calibration").to_frame())
    display(sel[["cal_max_residual_s", "resid_med_s", "resid_p95_s", "eps_F_s",
                 "av_lag_s", "av_lag_brady_s"]].describe())
    display(sel["qa_reason"].str.split("+").str[0].value_counts().rename("first QA reason")
            .to_frame())
pclock = local_json(config.DATA_DIR / "controls" / "p-clock.json")
show(plots.fig_p_clock(pclock) if pclock else None, "p_clock")

## 4. Causal contract

In [ ]:
CONTRACT_TESTS = ["tests/test_causal.py", "tests/test_labels.py", "tests/test_metrics.py",
                  "tests/test_masking.py", "tests/test_pcw.py", "tests/test_loader_allowlist.py",
                  "tests/test_gru_inputs.py", "tests/test_calibration_qa.py"]
if RUN_TESTS:
    out = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                          *CONTRACT_TESTS], cwd=PIPELINE, capture_output=True, text=True)
    print("\n".join(out.stdout.strip().splitlines()[-3:]))
from oncue import features as F
from oncue import loaders

print(f"{len(config.FEATURE_ALLOWLIST)} allowlist patterns; feature groups: "
      + ", ".join(sorted(F.GROUPS)))
print("provenance problems:", loaders.feature_provenance_problems(F.PROVENANCE) or "none")

## 5. Decision points: base rates and exclusions

In [ ]:
points = local("points")
if points is not None:
    dev = points[~points["in_vault"].astype(bool) & points["at_risk"].astype(bool)]
    rates = {c: float(dev[c].mean()) for c in config.LABEL_COLUMNS if c in dev}
    display(pd.Series(rates, name="base rate (at risk, dev)").to_frame())
    display(points["excl_reason"].fillna("at risk").value_counts().rename("points").to_frame())
elif results is not None:
    pooled = results[results["fold"].isna()]
    display(pooled.groupby(["target", "H", "g"])["base_rate"].first().to_frame())

## 6. Descriptives: peri-onset curves

In [ ]:
feats, eps = local("features"), local("hand_episodes")
if feats is not None and eps is not None:
    dev_ids = set(points.loc[~points["in_vault"].astype(bool), "class_id"]) if points is not None \
        else set(feats["class_id"])
    cur = report.peri_onset_curves(feats[feats["class_id"].isin(dev_ids)],
                                   report.onsets_from_bids(eps),
                                   ["q_open", "spk_silence_s", "pcw_unspent"])
    show(plots.fig_peri_onset(cur.dropna(subset=["mean"])), "peri_onset")
else:
    show(None, "peri_onset")

## 7. Models and learning curves

In [ ]:
from oncue import experiments

curves = experiments.tuning_curves()
if "M3" in curves:
    show(plots.fig_hgb_learning({k: v["curves"] for k, v in curves["M3"].items()},
                                {k: v["n_star"] for k, v in curves["M3"].items()}),
         "hgb_learning")
else:
    show(None, "hgb_learning")
for m in ("M2", "M4", "B1"):
    if m in curves:
        show(plots.fig_lr_c_curve({k: v["curves"] for k, v in curves[m].items()}, model=m),
             f"lr_c_{m}")
if results is not None:
    show(plots.fig_learning_curve(results) if results["condition"].str.startswith("LC-").any()
         else None, "learning_curve")

## 8. Comparison table and forest plot (H1-H5)

In [ ]:
if results is not None:
    display(plots.comparison_table(results))
oofs = bundle.oofs
segments = local("segments")
if LOCAL and config.OOF_DIR.exists():            # raw ids: H2 needs the segments table
    oofs = {p.stem: pd.read_parquet(p) for p in sorted(config.OOF_DIR.glob("*.parquet"))
            if not p.stem.endswith("_train")}
hyp = report.hypothesis_table(oofs, bundle.bids if not LOCAL else local("hand_episodes"),
                              segments=segments if LOCAL else None) if oofs else None
if hyp is not None:
    display(hyp)
    show(plots.fig_forest(hyp.dropna(subset=["estimate"])), "forest")

## 9. Event curves (FROC), time in warning and lead

In [ ]:
froc_bids = bundle.bids if bundle.bids is not None else local("hand_episodes")
curves = report.froc_curves(bundle.oofs, froc_bids, ("K0-M3", "K0-Bmin", "K0-R"))
ao = report.always_on_point(bundle.oofs["K0-AO"], froc_bids) if "K0-AO" in bundle.oofs else None
show(plots.fig_froc(curves, ao_point=ao) if curves else None, "froc")
if results is not None:
    cols = ["recall_far01", "realised_far01", "tiw_far01", "eventlift0_far01",
            "eventlift_shift_far01", "recall_far05", "realised_far05", "lead_med_s", "pafroc"]
    display(results[results["fold"].isna()].set_index("condition")[cols])

## 10. Timing decomposition (global and local shifts)

In [ ]:
show(plots.fig_timing_decomposition(results) if results is not None else None,
     "timing_decomposition")

## 11. Ablations and controls (P-plant, CALLON, IH+, N-name, A-mismatch)

In [ ]:
for cond in ("P-plant-M1", "P-plant-M2"):
    meta = local_json(config.OOF_DIR / f"{cond}.json")
    if meta and "plant" in meta:
        print(cond, meta["plant"])
callon = local_json(config.DATA_DIR / "controls" / "callon.json")
show(plots.fig_callon(callon) if callon else None, "callon")
nname = local_json(config.DATA_DIR / "controls" / "n-name.json")
show(plots.fig_name_rates(pd.DataFrame(nname["rates"])) if nname else None, "name_rates")
if results is not None:
    pooled = results[results["fold"].isna()].set_index("condition")
    ih = pooled[pooled.index.str.startswith(("IH", "N-"))]
    if len(ih):
        display(ih[["featureset", "ap_within", "ap"]])
    show(plots.fig_mismatch(results) if pooled.index.str.startswith("A-mismatch").any()
         else None, "mismatch")

## 12. Splits (leakage tax, time, leave-one-course-out)

In [ ]:
show(plots.fig_split_tax(results) if results is not None else None, "comparison_split_tax")

## 13. Sweeps and information horizon

In [ ]:
show(plots.fig_sweeps(results) if results is not None else None, "sweeps")
show(plots.fig_info_horizon(results) if results is not None else None, "info_horizon")

## 14. A-stream and F-punct

In [ ]:
stream = local_json(config.DATA_DIR / "controls" / "a-stream.json")
if stream:
    print({k: v for k, v in stream.items() if not isinstance(v, list)})
if results is not None:
    pooled = results[results["fold"].isna()].set_index("condition")
    rows = pooled[pooled["featureset"].isin(["F-all", "F-punct", "F-name"])
                  & pooled["model"].isin(["M2", "M3"])]
    display(rows[["model", "featureset", "ap_within", "ap"]])

## 15. Calibration

In [ ]:
if "K0-M3" in bundle.oofs:
    show(plots.fig_reliability(bundle.oofs["K0-M3"]), "reliability")
if results is not None:
    display(results[results["fold"].isna()].set_index("condition")[["brier", "brier_skill",
                                                                     "ece"]])

## 16. Vault

The vault (a hashed 20% of classes) is scored once, after G4, with
`python -m oncue train --open-vault`. Until then this section stays empty.

In [ ]:
pairs = {"V-M3-F-all": "K0-M3", "V-M3-F-all-me_hand": "F0-Fall-me_hand",
         "V-M3-F-struct": "F0-Fstruct", "V-R-F-q": "K0-R", "V-B1-F-struct": "K0-B1"}
has_vault = results is not None and results["condition"].isin(list(pairs)).any()
show(plots.fig_vault_vs_cv(results, pairs) if has_vault else None, "vault_vs_cv")

## 17. Aggregate error analysis

In [ ]:
if "K0-M3" in bundle.oofs:
    show(plots.fig_ap_within(bundle.oofs["K0-M3"]), "ap_within")
# A5 kNN explainer (could-tier, metadata only): heavy, so local and only with RERUN_MODELS
m3_local = config.OOF_DIR / "K0-M3.parquet"
if RERUN_MODELS and m3_local.exists():
    data = experiments.load_data(with_emb=True)
    show(plots.fig_knn(experiments.knn_summary(data, pd.read_parquet(m3_local))), "knn")
else:
    show(None, "knn")

## 18. Summary

In [ ]:
summary = report.summary(results, hyp, funnel)
if SAVE_FIGURES and summary:
    print("wrote", report.write_summary(config.FIGURES_DIR / "summary.json", summary))
display(pd.DataFrame(summary.get("conditions", {})).T)